In [ ]:
# ----------------------------------------------------------------------------
# 1. ADVANCED TOOL CREATION - CURRENCY CONVERSION
# ----------------------------------------------------------------------------


In [ ]:
from langchain_core.tools import tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests



# Define the currency conversion tool - 1
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Fetches the currency conversion factor between two currencies.
    Uses the ExchangeRate-API to get real-time conversion rates.
    
    Args:
        base_currency (str): Base currency code (e.g., 'USD', 'INR')
        target_currency (str): Target currency code (e.g., 'USD', 'EUR')
    
    Returns:
        dict: JSON response containing conversion rate and metadata
    """
    url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()


# Define the currency conversion tool - 2
@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Converts currency value using a given conversion rate.
    
    Args:
        base_currency_value (int): Amount in base currency
        conversion_rate (float): Conversion rate (injected from previous tool call)
    
    Returns:
        float: Converted amount in target currency
    """
    return base_currency_value * conversion_rate


In [ ]:
# Inspect convert tool arguments
print("\n--- Currency conversion tool arguments ---")
print(convert.args)

In [ ]:
# Test tools individually
print("\n--- Testing get_conversion_factor ---")
rate_data = get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})
print(rate_data)

print("\n--- Testing convert ---")
converted_value = convert.invoke({'base_currency_value': 10, 'conversion_rate': 85.16})
print(converted_value)

In [ ]:
# ----------------------------------------------------------------------------
# 2. MULTI-TOOL WORKFLOW - CHAINING TOOLS TOGETHER
# ----------------------------------------------------------------------------


In [ ]:
import json

# Reinitialize LLM with currency conversion tools
llm = ChatCohere(
    cohere_api_key=api_token,
    model="command-a-03-2025"
)

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])


In [ ]:
# Create query requiring both tools
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

print("\n--- Initial query ---")
print(messages)


In [ ]:
# Get AI response with tool calls
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

print("\n--- Tool calls requested by LLM ---")
print(ai_message.tool_calls)

In [ ]:
# Execute tools in sequence with data passing
for tool_call in ai_message.tool_calls:
    # Execute get_conversion_factor and extract rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # Parse JSON and extract conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # Append tool result to messages
        messages.append(tool_message1)
    
    # Execute convert tool using rate from previous tool
    if tool_call['name'] == 'convert':
        # Inject conversion rate into tool arguments
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)


In [ ]:
print("\n--- Messages after tool execution ---")
print(messages)

In [ ]:
# Get final response from LLM
final_answer = llm_with_tools.invoke(messages).content
print("\n--- Final conversion result ---")
print(final_answer)
